# Diversity-sampling experiment analysis

This notebook reads the `metrics.json` reports produced by the diversity-sampling experiment and rebuilds the three summary tables used in the experiment report:

1. recipe, AIG, and LUT diversity;
2. best-of-budget AIG and 6-LUT quality;
3. AIG-to-LUT Spearman rank correlations.

Every reported aggregate is computed across training seeds. Quality entries are the mean of the per-seed best values among 1,000 sampled trajectories; lower is better.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)

## Load and validate metric reports

Set `GFC_DIVERSITY_RESULTS` to analyze another result directory. Without it, the notebook searches the locations expected when it is launched either from the repository root or from `ntb/`.

In [ ]:
CIRCUITS = ["C5315", "dalu", "k2"]
ALGORITHMS = {
    "gflownet": "GFlowNet",
    "reinforce": "REINFORCE",
    "ppo": "PPO",
    "drills-a2c": "DRiLLS-A2C",
}

configured_root = os.environ.get("GFC_DIVERSITY_RESULTS")
candidates = [
    Path(configured_root).expanduser() if configured_root else None,
    Path("../gfc_outputs/bs_v0/diversity"),
    Path("../../gfc_outputs/bs_v0/diversity"),
]
RESULTS_ROOT = next((path.resolve() for path in candidates if path and path.is_dir()), None)
if RESULTS_ROOT is None:
    searched = "\n".join(f"  - {path}" for path in candidates if path)
    raise FileNotFoundError(
        "Could not find the diversity result directory. Set GFC_DIVERSITY_RESULTS. "
        f"Searched:\n{searched}"
    )

print(f"Reading reports from: {RESULTS_ROOT}")

In [ ]:
def flatten_group(circuit, algorithm_key, group):
    recipe = group["recipe_diversity"]
    quality = group["quality"]
    aig = quality["aig"]
    lut = quality["lut"]
    correlation = quality["aig_to_lut_rank_correlation"]
    return {
        "Circuit": circuit,
        "Algorithm": ALGORITHMS[algorithm_key],
        "algorithm_key": algorithm_key,
        "training_seed": group.get("training_seed", group["run_id"]),
        "run_id": group["run_id"],
        "num_samples": group["num_samples"],
        "unique_recipe_fraction": recipe["unique_recipe_fraction_curve"][-1]["mean"],
        "recipe_entropy_nats": recipe["recipe_entropy_nats"],
        "action_entropy_nats": recipe["action_entropy_nats"]["mean"],
        "aig_unique_pairs": group["aig_diversity"]["unique_size_depth_pairs"],
        "lut_unique_pairs": group["lut_diversity"]["unique_size_depth_pairs"],
        "aig_best_size": aig["best_size"]["value"],
        "aig_best_depth": aig["best_depth"]["value"],
        "aig_best_product": aig["best_size_depth_product"]["value"],
        "lut_best_size": lut["best_size"]["value"],
        "lut_best_depth": lut["best_depth"]["value"],
        "lut_best_product": lut["best_size_depth_product"]["value"],
        "lut_pareto_points": len(quality["lut_pareto_front"]),
        "size_rank_rho": correlation["size"],
        "depth_rank_rho": correlation["depth"],
    }

rows = []
report_rows = []
for circuit in CIRCUITS:
    for algorithm_key, algorithm_name in ALGORITHMS.items():
        report_path = RESULTS_ROOT / circuit / algorithm_key / "metrics.json"
        if not report_path.is_file():
            raise FileNotFoundError(f"Missing expected report: {report_path}")
        report = json.loads(report_path.read_text())
        if report.get("schema_version") != 1:
            raise ValueError(f"Unsupported schema in {report_path}: {report.get('schema_version')}")
        groups = report.get("groups", [])
        if not groups:
            raise ValueError(f"Report contains no seed groups: {report_path}")
        rows.extend(flatten_group(circuit, algorithm_key, group) for group in groups)
        report_rows.append({
            "Circuit": circuit,
            "Algorithm": algorithm_name,
            "Seed groups": len(groups),
            "Samples per seed": sorted({group["num_samples"] for group in groups}),
        })

seed_metrics = pd.DataFrame(rows)
seed_metrics["Circuit"] = pd.Categorical(seed_metrics["Circuit"], CIRCUITS, ordered=True)
seed_metrics["Algorithm"] = pd.Categorical(
    seed_metrics["Algorithm"], list(ALGORITHMS.values()), ordered=True
)
seed_metrics = seed_metrics.sort_values(["Circuit", "Algorithm", "training_seed"]).reset_index(drop=True)

display(Markdown(f"Loaded **{len(report_rows)} reports** and **{len(seed_metrics)} seed-level groups**."))
display(pd.DataFrame(report_rows))

## Diversity

`Unique recipes` is the unique-recipe fraction at the complete sampling budget. Action entropy is measured in nats. AIG/LUT pair counts are distinct `(size, depth)` outcomes and are shown as mean ± sample standard deviation across seeds.

Why gfn entropy $1.942$

In [ ]:
def mean_sd(series):
    return f"{series.mean():.1f} ± {series.std(ddof=1):.1f}"

diversity_rows = []
for (circuit, algorithm), frame in seed_metrics.groupby(
    ["Circuit", "Algorithm"], observed=True, sort=False
):
    diversity_rows.append({
        "Circuit": circuit,
        "Algorithm": algorithm,
        "Unique recipes": f"{100 * frame['unique_recipe_fraction'].mean():.2f}%",
        "Action entropy": f"{frame['action_entropy_nats'].mean():.3f}",
        "AIG pairs": mean_sd(frame["aig_unique_pairs"]),
        "LUT pairs": mean_sd(frame["lut_unique_pairs"]),
    })

diversity_table = pd.DataFrame(diversity_rows)
display(diversity_table.style.hide(axis="index"))

## Best-of-budget quality

Each cell is the mean across seeds of that seed's best result among its sampled trajectories. Products are `size × depth`. Bold cells mark the minimum within each circuit and metric; ties are all highlighted.

In [ ]:
QUALITY_COLUMNS = {
    "aig_best_size": "AIG size",
    "aig_best_depth": "AIG depth",
    "aig_best_product": "AIG product",
    "lut_best_size": "LUT size",
    "lut_best_depth": "LUT depth",
    "lut_best_product": "LUT product",
}

quality_table = (
    seed_metrics.groupby(["Circuit", "Algorithm"], observed=True, sort=False)
    [list(QUALITY_COLUMNS)]
    .mean()
    .rename(columns=QUALITY_COLUMNS)
    .reset_index()
)
quality_metric_columns = list(QUALITY_COLUMNS.values())

def highlight_per_circuit_minima(frame):
    styles = pd.DataFrame("", index=frame.index, columns=frame.columns)
    for indices in frame.groupby("Circuit", observed=True).groups.values():
        for column in quality_metric_columns:
            values = frame.loc[indices, column]
            styles.loc[indices, column] = np.where(
                np.isclose(values, values.min()),
                "font-weight: bold; background-color: #999999",
                "",
            )
    return styles

quality_styler = (
    quality_table.style
    .hide(axis="index")
    .format({column: "{:.1f}" for column in quality_metric_columns})
    .apply(highlight_per_circuit_minima, axis=None)
)
display(quality_styler)

## AIG-to-LUT rank correlation

Cells show `size ρ / depth ρ`, averaged across seeds. Positive Spearman ρ means that better-ranked AIG results tend to remain better-ranked after 6-LUT mapping. Undefined seed-level correlations are excluded from the corresponding mean.

In [ ]:
correlation_summary = (
    seed_metrics.groupby(["Circuit", "Algorithm"], observed=True, sort=False)
    .agg(
        size_rho=("size_rank_rho", "mean"),
        depth_rho=("depth_rank_rho", "mean"),
        size_n=("size_rank_rho", "count"),
        depth_n=("depth_rank_rho", "count"),
    )
    .reset_index()
)
correlation_summary["value"] = correlation_summary.apply(
    lambda row: f"{row['size_rho']:.2f} / {row['depth_rho']:.2f}", axis=1
)
correlation_table = (
    correlation_summary.pivot(index="Circuit", columns="Algorithm", values="value")
    .reindex(index=CIRCUITS, columns=list(ALGORITHMS.values()))
    .rename_axis(index=None, columns=None)
)
display(correlation_table)

incomplete = correlation_summary[
    (correlation_summary["size_n"] < correlation_summary[["size_n", "depth_n"]].max(axis=1))
    | (correlation_summary["depth_n"] < correlation_summary[["size_n", "depth_n"]].max(axis=1))
]
if not incomplete.empty:
    details = "; ".join(
        f"{row.Circuit} / {row.Algorithm}: size n={int(row.size_n)}, depth n={int(row.depth_n)}"
        for row in incomplete.itertuples()
    )
    display(Markdown(f"**Defined-correlation counts:** {details}."))